# Stacking & Blending Ensembles (by hand, then with sklearn)

A **stacked ensemble** trains several *diverse* base learners, then trains a second-level model — the **meta-learner** — to combine their predictions. The intuition: different model families make *different* mistakes (a linear model, a tree, a distance-based model, a naive-Bayes model all fail in different regions of feature space). If their errors are uncorrelated, a meta-learner can learn *when to trust which base model* and beat every one of them individually.

We build the two classic recipes from scratch and then confirm against the library:

1. **Stacking (with out-of-fold predictions)** — use `cross_val_predict` so every training row gets a base prediction produced by a model that *never saw that row*. This is the key to avoiding leakage.
2. **Blending (with a holdout set)** — a simpler variant: carve off a small holdout, fit bases on the rest, predict the holdout, train the meta-learner there. Less data-efficient, no cross-validation.
3. **`sklearn.ensemble.StackingClassifier`** — the batteries-included version, to check our hand-rolled stacker matches.

Everything runs on a small synthetic `make_classification` dataset so the whole notebook finishes in a few seconds.

In [ ]:
import numpy as np                                  # arrays / numeric glue for meta-feature matrices
import pandas as pd                                 # tidy leaderboard tables
import matplotlib.pyplot as plt                     # bar chart of the comparison

from sklearn.datasets import make_classification    # small synthetic binary-classification data
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold

# --- the DIVERSE base learners: deliberately different model families ---
from sklearn.linear_model import LogisticRegression # linear decision boundary
from sklearn.tree import DecisionTreeClassifier     # axis-aligned splits, high variance
from sklearn.neighbors import KNeighborsClassifier  # distance-based, local
from sklearn.svm import SVC                          # margin-based (needs probability=True for proba)
from sklearn.naive_bayes import GaussianNB           # generative, assumes feature independence
from sklearn.ensemble import RandomForestClassifier, StackingClassifier  # bagged trees + the library stacker

from sklearn.preprocessing import StandardScaler     # scale features (helps LogReg / KNN / SVC)
from sklearn.pipeline import make_pipeline           # bundle scaler+model so scaling is leak-free per fold
from sklearn.metrics import accuracy_score, roc_auc_score

RANDOM_STATE = 42                                    # one seed to make every split/model reproducible
np.random.seed(RANDOM_STATE)

## 1. Data

`make_classification` gives us a labelled matrix `X` of shape `(n_samples, n_features)` and a 0/1 target `y` of shape `(n_samples,)`. We add a few redundant/noise features so no single model type is trivially optimal — that diversity of failure modes is exactly what stacking exploits.

We make **three** logical partitions:

- **train** — used to fit base learners and (for stacking) to generate out-of-fold meta-features.
- **blend/holdout** — a slice carved *out of the training portion*, used only by the blending recipe.
- **test** — untouched until the very end, used to score every approach on equal footing.

In [ ]:
# 2000 rows, 20 features: 10 informative, 5 redundant (linear combos), 5 noise.
# class_sep controls how separable the classes are; a moderate value keeps the task non-trivial.
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_classes=2,
    class_sep=0.9,
    flip_y=0.03,            # 3% label noise -> keeps accuracy realistic, not 100%
    random_state=RANDOM_STATE,
)
print(f"X shape: {X.shape}, y shape: {y.shape}, positive rate: {y.mean():.3f}")

# First split: hold out 25% as the FINAL test set. stratify=y keeps the class ratio identical.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

# Second split: from the training portion, carve a 25% BLEND/HOLDOUT set for the blending recipe.
# Stacking will use the whole X_train_full via cross-validation; blending needs this separate slice.
X_tr, X_blend, y_tr, y_blend = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train_full
)

print(f"train_full: {X_train_full.shape}  (used whole for stacking OOF)")
print(f"  -> tr (base-fit for blending): {X_tr.shape}")
print(f"  -> blend (meta-fit for blending): {X_blend.shape}")
print(f"test (final scoring): {X_test.shape}")

## 2. The diverse base learners

Diversity is the whole point — stacking only helps if the base models disagree in useful ways. We pick six different families. The scale-sensitive ones (LogReg, KNN, SVC) are wrapped in a `StandardScaler` pipeline so that when we cross-validate, scaling statistics are computed **per fold** (no leakage). Tree/forest/NB are scale-invariant, so they get no scaler.

We keep a `factory` (a function returning a *fresh* model) for each learner, because both recipes need to refit clones repeatedly and we must never reuse an already-fitted estimator.

In [ ]:
# Each entry maps a name -> a zero-arg factory that returns a BRAND-NEW unfitted estimator.
# Using factories (not shared instances) guarantees every fold / recipe gets a clean model.
def base_factories():
    return {
        # LogisticRegression: linear boundary. Scaled because it is sensitive to feature magnitude.
        "logreg": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        # DecisionTree: high-variance, axis-aligned splits. Depth-capped to keep it from memorizing.
        "dtree":  lambda: DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
        # KNN: purely local / distance-based. Scaled so all features contribute equally to distance.
        "knn":    lambda: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
        # SVC with an RBF kernel. probability=True is REQUIRED so we can read predict_proba later.
        "svc":    lambda: make_pipeline(StandardScaler(), SVC(probability=True, random_state=RANDOM_STATE)),
        # GaussianNB: generative model assuming per-class Gaussian, independent features.
        "gnb":    lambda: GaussianNB(),
        # RandomForest: an ensemble itself (bagged trees) -> already strong, low variance.
        "rf":     lambda: RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_STATE, n_jobs=-1),
    }

BASE_NAMES = list(base_factories().keys())   # stable ordering used everywhere below
print("base learners:", BASE_NAMES)

## 3. Baseline: each base learner on its own

Before combining anything, fit every base learner on the full training set and score it on the test set. These numbers are the bar the ensembles must clear. We record both **accuracy** (threshold-0.5 correctness) and **ROC AUC** (ranking quality, threshold-independent) — a model can win on one and lose on the other.

In [ ]:
# results[name] = {"accuracy": ..., "roc_auc": ...}; we append the ensembles to this same dict later.
results = {}

# Keep the fitted base models around: predictions on the test set become meta-features for
# the FINAL stacking prediction (the meta-learner needs test-time base outputs too).
fitted_full = {}

for name, make in base_factories().items():
    model = make()                                  # fresh estimator
    model.fit(X_train_full, y_train_full)           # train on ALL training data
    proba = model.predict_proba(X_test)[:, 1]       # (n_test,) P(class=1); column 1 = positive class
    preds = (proba >= 0.5).astype(int)              # threshold to hard labels for accuracy
    results[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, proba),
    }
    fitted_full[name] = model                        # stash for later meta-feature construction
    print(f"{name:>7}  acc={results[name]['accuracy']:.4f}  auc={results[name]['roc_auc']:.4f}")

## 4. Stacking by hand — why out-of-fold predictions?

The naive (and **wrong**) way to build meta-features: fit each base learner on the training set, then ask it to predict *that same training set*, and feed those predictions to the meta-learner.

That leaks. A flexible base model (a deep tree, KNN with small *k*) can nearly **memorize** its training rows, so its in-sample predictions look almost perfect — far better than it will ever do on unseen data. The meta-learner then learns to trust that model blindly, and the whole stack overfits and collapses on the test set.

**Out-of-fold (OOF) predictions fix this.** Split the training data into $K$ folds. For each fold, train the base model on the *other* $K-1$ folds and predict the held-out fold. Stitch the held-out predictions back together and every training row now carries a prediction from a model that **never saw it** — an honest, leakage-free estimate of that base model's behaviour on unseen data. `cross_val_predict` does exactly this stitching for us.

So the meta-features have shape `(n_train, n_base_learners)`: one out-of-fold probability column per base learner.

In [ ]:
# Shared fold scheme so every base learner is cross-validated on the SAME partition -> its OOF
# columns are aligned row-for-row and comparable.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

n_train = X_train_full.shape[0]
n_bases = len(BASE_NAMES)

# Meta-feature matrix for TRAINING the meta-learner. One column per base learner.
# Shape: (n_train, n_bases). Each column is filled with leakage-free OOF probabilities.
oof_meta = np.zeros((n_train, n_bases))

for j, name in enumerate(BASE_NAMES):
    model = base_factories()[name]()                 # fresh estimator for the CV loop
    # cross_val_predict with method='predict_proba' returns, for every training row, the
    # positive-class probability produced by the fold in which that row was held out.
    # Output shape: (n_train, 2); we keep column 1 = P(class=1).
    oof = cross_val_predict(model, X_train_full, y_train_full, cv=cv, method="predict_proba", n_jobs=-1)
    oof_meta[:, j] = oof[:, 1]                        # fill this base learner's OOF column
    print(f"OOF built for {name:>7}  ->  column {j}")

print(f"\noof_meta shape: {oof_meta.shape}  (rows = train samples, cols = base learners)")

### Train the meta-learner, then predict the test set

The meta-learner is a plain `LogisticRegression`: it learns a weighted combination of the base probabilities (effectively *how much to trust each base model*). We train it on the OOF meta-features.

For the **final test prediction** we need test-time meta-features. Those come from the base models **refit on the full training set** (the `fitted_full` we saved earlier) predicting the test set — giving a `(n_test, n_bases)` matrix that we push through the trained meta-learner.

In [ ]:
# Meta-learner: simple, well-regularized linear combiner of the base probabilities.
meta_learner = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
meta_learner.fit(oof_meta, y_train_full)             # learn on leakage-free OOF meta-features

# Build TEST meta-features from the base models already fitted on the full training set.
# Column j = base learner j's P(class=1) on the test rows. Shape: (n_test, n_bases).
test_meta = np.column_stack([
    fitted_full[name].predict_proba(X_test)[:, 1] for name in BASE_NAMES
])
print(f"test_meta shape: {test_meta.shape}")

# Meta-learner turns the base probabilities into the final stacked prediction.
stack_proba = meta_learner.predict_proba(test_meta)[:, 1]
stack_preds = (stack_proba >= 0.5).astype(int)
results["STACK (manual)"] = {
    "accuracy": accuracy_score(y_test, stack_preds),
    "roc_auc": roc_auc_score(y_test, stack_proba),
}
print(f"\nSTACK (manual)  acc={results['STACK (manual)']['accuracy']:.4f}  auc={results['STACK (manual)']['roc_auc']:.4f}")

# Inspect the learned meta-weights: which base learners does the stacker lean on?
for name, coef in zip(BASE_NAMES, meta_learner.coef_[0]):
    print(f"  meta weight  {name:>7}: {coef:+.3f}")

## 5. Blending by hand — how it differs from stacking

Blending is the *poor-man's* stacking. Instead of cross-validation, you just carve off a single **holdout (blend) set** up front:

1. Fit the base learners on the training part only (`X_tr`, **not** the blend set).
2. Have them predict the **blend set** — those predictions are the meta-features, and they are leakage-free because the bases never trained on the blend rows.
3. Train the meta-learner on those blend-set meta-features.

**Trade-offs vs stacking:**

| | Stacking (OOF) | Blending (holdout) |
|---|---|---|
| Meta-features from | K-fold cross_val_predict | single holdout split |
| Data efficiency | high — every train row becomes a meta-feature | low — the blend set is spent on the meta-learner only |
| Base fit for meta | many refits (K per learner) | one fit per learner |
| Variance of meta-features | lower (averaged over folds) | higher (one split) |
| Complexity / speed | more code, slower | simpler, faster |

Blending is easier to implement and avoids any subtle CV leakage bugs, but it wastes data (the blend rows never train the base learners for the final model) and its meta-features are noisier because they rest on a single split.

In [ ]:
# Step 1: fit each base learner on the TRAIN-ONLY part (X_tr), never touching the blend rows.
fitted_tr = {}
for name, make in base_factories().items():
    m = make()
    m.fit(X_tr, y_tr)                                 # base learners see only X_tr / y_tr
    fitted_tr[name] = m

# Step 2: predict the BLEND set -> leakage-free meta-features (bases never saw these rows).
# Shape: (n_blend, n_bases).
blend_meta = np.column_stack([
    fitted_tr[name].predict_proba(X_blend)[:, 1] for name in BASE_NAMES
])
print(f"blend_meta shape: {blend_meta.shape}  (rows = blend samples, cols = base learners)")

# Step 3: train the meta-learner on the blend-set meta-features and their true labels.
blend_meta_learner = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
blend_meta_learner.fit(blend_meta, y_blend)

# For the final TEST prediction we reuse the SAME base models (fit on X_tr) to build test
# meta-features. (One could optionally refit bases on tr+blend; we keep it simple and honest.)
blend_test_meta = np.column_stack([
    fitted_tr[name].predict_proba(X_test)[:, 1] for name in BASE_NAMES
])
blend_proba = blend_meta_learner.predict_proba(blend_test_meta)[:, 1]
blend_preds = (blend_proba >= 0.5).astype(int)
results["BLEND (manual)"] = {
    "accuracy": accuracy_score(y_test, blend_preds),
    "roc_auc": roc_auc_score(y_test, blend_proba),
}
print(f"\nBLEND (manual)  acc={results['BLEND (manual)']['accuracy']:.4f}  auc={results['BLEND (manual)']['roc_auc']:.4f}")

## 6. The library equivalent — `StackingClassifier`

`sklearn.ensemble.StackingClassifier` does exactly what our manual stacker does: it uses internal cross-validation (`cv=`) to generate out-of-fold meta-features for the final estimator, and by default refits the base learners on the full training data for test-time prediction. We give it the same six bases, the same `LogisticRegression` meta-learner, and the same 5-fold CV, then confirm the score lands right next to our hand-rolled `STACK (manual)`.

Small numeric differences are expected (fold shuffling internals, `stack_method` defaults), but they should be in the same ballpark — that's our correctness check.

In [ ]:
# estimators must be a list of (name, estimator) tuples using FRESH factory instances.
estimators = [(name, base_factories()[name]()) for name in BASE_NAMES]

sk_stack = StackingClassifier(
    estimators=estimators,                                  # the diverse base learners
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),  # meta-learner
    stack_method="predict_proba",                           # feed probabilities (matches our manual version)
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),        # same OOF scheme
    passthrough=False,                                      # meta-learner sees ONLY base outputs (like ours)
    n_jobs=-1,
)
sk_stack.fit(X_train_full, y_train_full)                    # handles OOF + base refit internally

sk_proba = sk_stack.predict_proba(X_test)[:, 1]
sk_preds = (sk_proba >= 0.5).astype(int)
results["STACK (sklearn)"] = {
    "accuracy": accuracy_score(y_test, sk_preds),
    "roc_auc": roc_auc_score(y_test, sk_proba),
}
print(f"STACK (sklearn) acc={results['STACK (sklearn)']['accuracy']:.4f}  auc={results['STACK (sklearn)']['roc_auc']:.4f}")
print(f"STACK (manual)  acc={results['STACK (manual)']['accuracy']:.4f}  auc={results['STACK (manual)']['roc_auc']:.4f}")

# Correctness check: the two stackers should agree to within a small tolerance.
diff = abs(results['STACK (sklearn)']['roc_auc'] - results['STACK (manual)']['roc_auc'])
print(f"\n|AUC difference| between manual and sklearn stacker: {diff:.4f}  -> {'MATCH' if diff < 0.03 else 'CHECK'}")

## 7. Leaderboard & comparison

Finally, line up every individual base learner against the stacked and blended ensembles on both metrics. We print a sorted leaderboard (by ROC AUC) and draw a grouped bar chart. If stacking is doing its job, the ensemble rows sit at (or above) the top of the pack — combining diverse learners should match or beat the single best base model.

In [ ]:
# Turn the results dict into a tidy DataFrame and sort by ROC AUC (then accuracy).
board = (
    pd.DataFrame(results).T                          # rows = models, cols = metrics
      .rename_axis("model")
      .sort_values(["roc_auc", "accuracy"], ascending=False)
)
print("=== Leaderboard (sorted by ROC AUC) ===")
print(board.round(4).to_string())

In [ ]:
# Grouped bar chart: accuracy and ROC AUC side by side for every model.
# Ensembles are highlighted in a warm colour so they stand out from the base learners.
labels = board.index.tolist()                        # model names in leaderboard order
x = np.arange(len(labels))                            # one x slot per model
width = 0.4                                           # width of each bar within a group

# Colour ensembles (names containing STACK/BLEND) differently from plain base learners.
is_ens = [("STACK" in n or "BLEND" in n) for n in labels]
acc_colors = ["#cc6666" if e else "#6699cc" for e in is_ens]
auc_colors = ["#a24545" if e else "#3f6fa1" for e in is_ens]

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - width/2, board["accuracy"], width, label="accuracy", color=acc_colors)
b2 = ax.bar(x + width/2, board["roc_auc"],  width, label="roc_auc",  color=auc_colors)

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylim(0.5, 1.0)                                 # zoom into the meaningful range
ax.set_ylabel("score")
ax.set_title("Base learners vs Stacked vs Blended ensembles")
ax.legend(loc="lower left")

# Annotate each bar with its value for quick reading.
for bars in (b1, b2):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.005, f"{h:.3f}",
                ha="center", va="bottom", fontsize=7, rotation=90)

plt.tight_layout()
plt.show()

## 8. Takeaways

- **Stacking** learns a data-driven combination of *diverse* base learners; the meta-learner discovers how much to trust each one. The essential trick is **out-of-fold predictions** (`cross_val_predict`) so the meta-learner trains on honest, leakage-free base outputs.
- **Blending** is the simpler cousin: one holdout split instead of K-fold CV. Easier and faster, but it spends data on the blend set and its meta-features are noisier.
- Our **hand-rolled stacker matched `StackingClassifier`** to within a small tolerance — confirming we understood what the library does under the hood.
- Ensembling pays off most when the base learners are genuinely diverse *and* individually decent. If one base model dominates and the rest are weak/correlated, the meta-learner mostly just copies the strong one and the gains shrink.